# 🌍 NewsPulse — Spark Analysis
**Ryan Adya Purwanto (5027231046)**

Analisis 3 pertanyaan bisnis dari data berita yang tersimpan di HDFS:
1. Kata paling sering muncul di judul berita
2. Distribusi berita per sumber
3. Volume publikasi per jam

In [ ]:
# Ryan Adya Purwanto — Setup
!pip install pyspark -q

In [ ]:
# Ryan Adya Purwanto — Inisialisasi SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json
import os

spark = SparkSession.builder \
    .appName("NewsPulse Analysis") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("SparkSession siap!")

In [ ]:
# Ryan Adya Purwanto — Load Data dari HDFS
print("Loading data dari HDFS...")

# Baca semua file JSON dari HDFS
df_api = spark.read.option("multiLine", True).json("hdfs://namenode:8020/data/news/api/")
df_rss = spark.read.option("multiLine", True).json("hdfs://namenode:8020/data/news/rss/")

# Gabungkan semua berita
# Samakan schema dulu (pilih kolom yang sama)
cols = ["title", "source", "published_at", "fetched_at", "url", "description"]

df_api_sel = df_api.select(*[F.col(c) if c in df_api.columns else F.lit(None).alias(c) for c in cols])
df_rss_sel = df_rss.select(*[F.col(c) if c in df_rss.columns else F.lit(None).alias(c) for c in cols])

df_all = df_api_sel.union(df_rss_sel).dropDuplicates(["url"]).filter(F.col("title").isNotNull())

print(f"Total artikel: {df_all.count():,}")
print(f"  - dari API : {df_api.count():,}")
print(f"  - dari RSS : {df_rss.count():,}")
df_all.show(5, truncate=60)

In [ ]:
# Ryan Adya Purwanto — Register sebagai SQL View
df_all.createOrReplaceTempView("berita")
print("View 'berita' siap untuk Spark SQL!")

## Analisis 1 — Kata Paling Sering di Judul Berita
Menggunakan `split()` dan `explode()` untuk memecah judul menjadi kata-kata,
lalu filter stopwords dan hitung frekuensi.

In [ ]:
# Ryan Adya Purwanto — Analisis 1: Top Words
stopwords = [
    "dan", "yang", "di", "ke", "dari", "untuk", "dengan",
    "ini", "itu", "pada", "dalam", "adalah", "akan", "tidak",
    "ada", "telah", "juga", "lebih", "sudah", "saat", "bisa",
    "oleh", "atas", "atau", "karena", "jika", "tapi", "bagi",
    "para", "pun", "hal", "nya", "kami", "kita", "mereka",
    "the", "a", "an", "of", "in", "to", "and", "for"
]

result_words = spark.sql(f"""
    SELECT word, COUNT(*) AS count
    FROM (
        SELECT explode(split(lower(title), '[^a-zA-Z0-9äöüÄÖÜ]+')) AS word
        FROM berita
        WHERE title IS NOT NULL
    )
    WHERE word NOT IN ({','.join([f"'{w}'" for w in stopwords])})
      AND length(word) > 3
    GROUP BY word
    ORDER BY count DESC
    LIMIT 20
""")

print("=" * 40)
print("  TOP 20 KATA DI JUDUL BERITA")
print("=" * 40)
result_words.show(20, truncate=False)

print("\n📌 Interpretasi:")
print("Kata-kata di atas mencerminkan isu yang paling banyak dibicarakan media nasional hari ini.")
print("Semakin tinggi frekuensi, semakin banyak artikel yang membahas topik tersebut.")

## Analisis 2 — Distribusi Berita per Sumber
Menghitung jumlah artikel dari setiap sumber media menggunakan `groupBy`.

In [ ]:
# Ryan Adya Purwanto — Analisis 2: Distribusi per Sumber
result_sources = spark.sql("""
    SELECT
        source,
        COUNT(*) AS count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS persen
    FROM berita
    WHERE source IS NOT NULL
    GROUP BY source
    ORDER BY count DESC
""")

print("=" * 50)
print("  DISTRIBUSI BERITA PER SUMBER")
print("=" * 50)
result_sources.show(truncate=False)

print("\n📌 Interpretasi:")
print("Distribusi ini menunjukkan kontribusi tiap sumber media dalam pipeline.")
print("Ketidakseimbangan bisa terjadi jika satu sumber update lebih sering dari yang lain.")

## Analisis 3 — Volume Publikasi per Jam
Menghitung berapa banyak berita dipublikasikan di setiap jam menggunakan `HOUR()`.

In [ ]:
# Ryan Adya Purwanto — Analisis 3: Volume per Jam
result_hours = spark.sql("""
    SELECT
        HOUR(TO_TIMESTAMP(published_at)) AS hour,
        COUNT(*) AS count
    FROM berita
    WHERE published_at IS NOT NULL
      AND published_at != ''
    GROUP BY hour
    ORDER BY hour
""")

print("=" * 40)
print("  VOLUME BERITA PER JAM")
print("=" * 40)
result_hours.show(24, truncate=False)

# Temukan jam tersibuk
busiest = result_hours.orderBy(F.desc("count")).first()
print(f"\n📌 Interpretasi:")
print(f"Jam tersibuk: {busiest['hour']}:00 WIB dengan {busiest['count']} artikel.")
print("Pola ini membantu PR agency menentukan kapan harus memantau media secara intensif.")

In [ ]:
# Ryan Adya Purwanto — Simpan Hasil ke HDFS & Dashboard JSON
import json

# Simpan ke HDFS
result_words.write.mode("overwrite").json("hdfs://namenode:8020/data/news/hasil/top_words")
result_sources.write.mode("overwrite").json("hdfs://namenode:8020/data/news/hasil/sources")
result_hours.write.mode("overwrite").json("hdfs://namenode:8020/data/news/hasil/hours")

print("✅ Hasil tersimpan ke HDFS: /data/news/hasil/")

# Simpan juga ke dashboard/data/spark_results.json
words_list   = [{"word": r["word"],   "count": r["count"]}   for r in result_words.collect()]
sources_list = [{"source": r["source"], "count": r["count"], "persen": float(r["persen"])} for r in result_sources.collect()]
hours_list   = [{"hour": r["hour"],   "count": r["count"]}   for r in result_hours.collect() if r["hour"] is not None]

spark_results = {
    "top_words":    words_list,
    "sources":      sources_list,
    "hours":        hours_list,
    "generated_at": __import__('datetime').datetime.now().isoformat()
}

# Simpan ke file lokal untuk dashboard
output_path = "../dashboard/data/spark_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(spark_results, f, ensure_ascii=False, indent=2)

print(f"✅ Hasil tersimpan ke dashboard: {output_path}")
print(f"   - {len(words_list)} kata trending")
print(f"   - {len(sources_list)} sumber berita")
print(f"   - {len(hours_list)} slot jam")

## Ringkasan Analisis

| Analisis | Metode | Insight |
|---|---|---|
| Kata trending | `split() + explode() + filter stopwords` | Topik paling hangat hari ini |
| Distribusi sumber | `groupBy source + count` | Kontribusi tiap media |
| Volume per jam | `HOUR(TO_TIMESTAMP) + groupBy` | Jam paling aktif publikasi berita |

**Jawaban pertanyaan bisnis:**
> Kata-kata dengan frekuensi tertinggi mencerminkan topik paling hangat. 
> Jam tersibuk menunjukkan kapan PR agency harus paling waspada memantau media.